# NB_lab_test_trends

Builds `adb_healthcare.gold.lab_test_trends` (one row per test_name per calendar month) from `silver.lab_results`, enriched with month and quarter names from `silver.calendar`, per the STTM workbook `sttm_silver_to_gold.xlsx`. Full rebuild via CREATE OR REPLACE TABLE, with LAG-based month-over-month abnormal rate trends.

In [0]:
%sql
CREATE OR REPLACE TABLE adb_healthcare.gold.lab_test_trends AS
WITH monthly AS (
  SELECT
    test_name,
    year(test_date) AS year,
    month(test_date) AS month,
    COUNT(*) AS total_tests,
    COUNT_IF(result_status = 'Abnormal') AS abnormal_test_count,
    COUNT_IF(result_status = 'Critical') AS critical_test_count
  FROM adb_healthcare.silver.lab_results
  GROUP BY test_name, year(test_date), month(test_date)
),
rated AS (
  SELECT
    *,
    CAST(abnormal_test_count * 100.0 / total_tests AS DECIMAL(5,2)) AS abnormal_rate_pct,
    CAST(critical_test_count * 100.0 / total_tests AS DECIMAL(5,2)) AS critical_rate_pct
  FROM monthly
),
trended AS (
  SELECT
    *,
    LAG(abnormal_rate_pct) OVER (PARTITION BY test_name ORDER BY year, month) AS prev_month_abnormal_rate_pct
  FROM rated
)
SELECT
  t.test_name,
  CAST(t.year AS INT) AS year,
  CAST(t.month AS INT) AS month,
  c.month_name,
  CONCAT('Q', c.quarter) AS quarter_name,
  CAST(t.total_tests AS INT) AS total_tests,
  CAST(t.abnormal_test_count AS INT) AS abnormal_test_count,
  CAST(t.critical_test_count AS INT) AS critical_test_count,
  t.abnormal_rate_pct,
  t.critical_rate_pct,
  CAST(t.prev_month_abnormal_rate_pct AS DECIMAL(5,2)) AS prev_month_abnormal_rate_pct,
  CAST(t.abnormal_rate_pct - t.prev_month_abnormal_rate_pct AS DECIMAL(6,2)) AS abnormal_rate_change_pct
FROM trended t
LEFT JOIN adb_healthcare.silver.calendar c
  ON c.date_key = CAST(date_format(make_date(t.year, t.month, 1), 'yyyyMMdd') AS INT);

In [0]:
%sql
WITH expected AS (
  SELECT * FROM VALUES
    (1,'test_name','string'),(2,'year','int'),(3,'month','int'),(4,'month_name','string'),
    (5,'quarter_name','string'),(6,'total_tests','int'),(7,'abnormal_test_count','int'),
    (8,'critical_test_count','int'),(9,'abnormal_rate_pct','decimal(5,2)'),(10,'critical_rate_pct','decimal(5,2)'),
    (11,'prev_month_abnormal_rate_pct','decimal(5,2)'),(12,'abnormal_rate_change_pct','decimal(6,2)')
  AS t(position, column_name, expected_type)
),
actual AS (
  SELECT ordinal_position + 1 AS position, column_name,
         lower(CASE WHEN data_type = 'DECIMAL' THEN concat('decimal(', numeric_precision, ',', numeric_scale, ')') ELSE data_type END) AS actual_type
  FROM adb_healthcare.information_schema.columns
  WHERE table_schema = 'gold' AND table_name = 'lab_test_trends'
)
SELECT
  COALESCE(e.position, a.position) AS position,
  COALESCE(e.column_name, a.column_name) AS column_name,
  e.expected_type,
  a.actual_type,
  CASE WHEN e.column_name IS NULL THEN 'UNEXPECTED COLUMN'
       WHEN a.column_name IS NULL THEN 'MISSING'
       WHEN e.expected_type <> a.actual_type THEN 'TYPE MISMATCH'
       WHEN e.position <> a.position THEN 'ORDER MISMATCH'
       ELSE 'OK' END AS status
FROM expected e
FULL OUTER JOIN actual a ON e.column_name = a.column_name
ORDER BY position;

In [0]:
%sql
SELECT
  COUNT(*) AS gold_rows,
  COUNT(DISTINCT test_name, year, month) AS distinct_grain_keys,
  COUNT(DISTINCT test_name) AS test_types,
  MIN(make_date(year, month, 1)) AS first_month,
  MAX(make_date(year, month, 1)) AS last_month,
  SUM(total_tests) AS gold_total_tests,
  (SELECT COUNT(*) FROM adb_healthcare.silver.lab_results) AS silver_total_tests,
  SUM(abnormal_test_count) AS gold_abnormal,
  (SELECT COUNT_IF(result_status = 'Abnormal') FROM adb_healthcare.silver.lab_results) AS silver_abnormal,
  SUM(critical_test_count) AS gold_critical,
  (SELECT COUNT_IF(result_status = 'Critical') FROM adb_healthcare.silver.lab_results) AS silver_critical,
  COUNT_IF(month_name IS NULL OR quarter_name IS NULL) AS missing_calendar_enrichment
FROM adb_healthcare.gold.lab_test_trends;

In [0]:
%sql
WITH chk AS (
  SELECT
    *,
    ROW_NUMBER() OVER (PARTITION BY test_name ORDER BY year, month) AS rn,
    LAG(make_date(year, month, 1)) OVER (PARTITION BY test_name ORDER BY year, month) AS prev_month_start,
    LAG(abnormal_rate_pct) OVER (PARTITION BY test_name ORDER BY year, month) AS expected_prev_rate,
    CAST(abnormal_test_count * 100.0 / total_tests AS DECIMAL(5,2)) AS expected_abn_rate,
    CAST(critical_test_count * 100.0 / total_tests AS DECIMAL(5,2)) AS expected_crit_rate
  FROM adb_healthcare.gold.lab_test_trends
)
SELECT
  COUNT_IF(abnormal_rate_pct <> expected_abn_rate) AS abnormal_rate_mismatches,
  COUNT_IF(critical_rate_pct <> expected_crit_rate) AS critical_rate_mismatches,
  COUNT_IF(rn = 1 AND prev_month_abnormal_rate_pct IS NOT NULL) AS first_month_prev_not_null,
  COUNT_IF(rn > 1 AND prev_month_abnormal_rate_pct IS DISTINCT FROM expected_prev_rate) AS prev_rate_mismatches,
  COUNT_IF(abnormal_rate_change_pct IS DISTINCT FROM CAST(abnormal_rate_pct - prev_month_abnormal_rate_pct AS DECIMAL(6,2))) AS change_mismatches,
  COUNT_IF(rn = 1) AS first_month_rows,
  COUNT_IF(prev_month_abnormal_rate_pct IS NULL) AS null_prev_rows,
  COUNT_IF(rn > 1 AND months_between(make_date(year, month, 1), prev_month_start) > 1) AS rows_after_month_gap,
  MIN(abnormal_rate_change_pct) AS min_change_pp,
  MAX(abnormal_rate_change_pct) AS max_change_pp
FROM chk;

In [0]:
# DBTITLE 1,Return lab test trends record count
updated_df = spark.table("adb_healthcare.gold.lab_test_trends")
record_count = updated_df.count()
dbutils.notebook.exit(str(record_count))